# 🚀 Lab 5A — Why GPUs Make AI Fast
**Foothold · GPU Computing for AI · Unit 5 (first half) · Book: Step 5, Sections 5.1–5.5**

Measure the tricks that make GPUs so good at AI: tiling, Tensor Cores and low precision, the roofline, and batch size. Then put them together to speed up a real training run.

| Part | What you do | Goes with Foothold lesson |
|---|---|---|
| A | Naive vs tiled matrix-multiply kernels vs cuBLAS | 5.1 Parallel Matrix Multiply & Tiling |
| B | FP32 vs FP16: speed, error, overflow, mixed-precision training | 5.2 Precision |
| C | Measure your GPU's roof and plot your own roofline | 5.3 Compute-bound vs Memory-bound |
| D | CPU vs GPU training, batch-size sweep, memory | 5.4 Batch Size & Training on GPU |
| E | The speed-up ladder on Fashion-MNIST | 5.5 Capstone |

⏱️ About 2 hours (A–C, then D–E). 📘 Textbook: Kirk & Hwu Ch 5 and Ch 16 · Hamacher Ch 9 (floating point) · H&P Ch 4 (roofline). Course site: https://aishugithub.github.io/course_GPU/

💬 **Every line of code has a comment** (after `#` in Python, after `//` inside the CUDA C kernels).

## ▶️ How to run this notebook

1. **Save your own copy first:** menu **File → Save a copy in Drive**. Work in that copy — the original is read-only, and your copy is what you submit.
2. **Run cells in order, top to bottom.** Click a grey code cell and press **Shift + Enter** (or the ▶ button on its left). Wait for it to finish (a ✓ and the output appear) before running the next one.
3. **Switch on the GPU:** menu **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**. (This lab needs it. If Colab says no GPU is available, wait a while and try again — the free tier has limits.)
4. **Lost or confused?** Menu **Runtime → Restart session and run all** starts everything again from a clean slate.
5. Cells marked **✏️ YOUR TURN** are exercises: change the code, run it again, and write your answer in the text cell below it (double-click a text cell to edit it).

## 0 · Setup — run both cells

In [ ]:
import time, subprocess            # time: stopwatch; subprocess: run system commands
import numpy as np                  # arrays and maths on the CPU
import matplotlib.pyplot as plt     # charts
import torch                        # PyTorch: runs maths on the CPU or the GPU
from torch import nn                # nn: PyTorch's layers and losses

if torch.cuda.is_available():       # is there an NVIDIA GPU we can use? ("cuda" = NVIDIA's GPU system)
    device = "cuda"                 # yes: from now on, device = "cuda" means "put it on the GPU"
    print("✅ GPU found:", torch.cuda.get_device_name(0))   # the name of GPU number 0 (the first one)
else:                               # no GPU in this session
    device = "cpu"                  # fall back to the CPU
    print("⚠️ NO GPU! Go to Runtime → Change runtime type → T4 GPU → Save, then run this cell again.")   # how to get one
    print("   (Most of this notebook needs the GPU, so switch it on before going further.)")              # and why it matters

def sync():                         # GPU work runs in the background: wait for it to really finish before reading the clock
    if device == "cuda":            # (only needed when there is a GPU)
        torch.cuda.synchronize()    # wait until every queued GPU job is done

def timeit(fn, repeats=5):          # run fn several times and return the average time of one run, in seconds
    fn(); sync()                    # warm-up run (the first GPU call has extra start-up cost)
    t0 = time.perf_counter()        # start the stopwatch
    for _ in range(repeats):        # run it several times
        fn()                        # the work itself
    sync()                          # wait until the GPU has really finished all the runs
    return (time.perf_counter() - t0) / repeats   # total time ÷ number of runs

In [ ]:
# CuPy lets Python compile and run small CUDA C kernels. It comes pre-installed on Colab.
try:                                # "try": attempt this; if it fails, jump to "except" instead of crashing
    import cupy as cp               # CuPy: NumPy-like arrays on the GPU, plus our own CUDA kernels
    HAVE_CUPY = device == "cuda"    # we can only use it if there is also a GPU
    print("CuPy", cp.__version__, "ready" if HAVE_CUPY else "(installed, but no GPU in this session)")   # say whether it's usable
except ImportError:                 # CuPy is not installed
    HAVE_CUPY = False               # remember that, so later cells can say so
    print("CuPy not found. On Colab with a GPU, run:  !pip install cupy-cuda12x   then re-run this cell.")   # how to install it

def cuda_time_ms(launch, repeats=5):                # time GPU work with the GPU's own clock (CUDA events); returns ms per run
    launch(); cp.cuda.Device().synchronize()        # warm-up run, then wait for it to finish
    start, end = cp.cuda.Event(), cp.cuda.Event()   # two time-stamps on the GPU
    start.record()                                  # stamp "start" in the GPU's queue
    for _ in range(repeats):                        # several runs
        launch()                                    # queue the kernel once more
    end.record(); end.synchronize()                 # stamp "end", and wait until the GPU reaches it
    return cp.cuda.get_elapsed_time(start, end) / repeats   # time between the stamps ÷ runs

---
## Part A · Naive vs Tiled vs cuBLAS  *(Unit 5.1)*

Two hand-written kernels from the lesson, compiled with CuPy, raced against PyTorch's `A @ B` (which calls NVIDIA's cuBLAS library) in FP32 and FP16.

In [ ]:
# two CUDA C kernels in one string (inside the C code, comments start with //)
mm_src = r'''
#define T 16   // the tile size: each block works on 16 × 16 tiles
extern "C" __global__   // a kernel, launched from the CPU
void matmul_naive(const float *A, const float *B, float *C, int n) {   // C = A × B, all n × n
    int row = blockIdx.y * blockDim.y + threadIdx.y;   // which output row this thread computes
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // and which column
    if (row < n && col < n) {   // the safety check
        float sum = 0.0f;   // the running total for this cell
        for (int k = 0; k < n; k++)   // walk along row of A and down column of B...
            sum += A[row * n + k] * B[k * n + col];   // ...reading BOTH from slow global memory every time
        C[row * n + col] = sum;   // write the finished cell
    }
}

extern "C" __global__   // the tiled version
void matmul_tiled(const float *A, const float *B, float *C, int n) {   // the same job, with tiles
    __shared__ float As[T][T];   // a tile of A in fast shared memory, shared by the whole block
    __shared__ float Bs[T][T];   // and a tile of B
    int row = blockIdx.y * T + threadIdx.y;   // this thread's output row
    int col = blockIdx.x * T + threadIdx.x;   // and column
    float sum = 0.0f;   // the running total for this cell
    for (int p = 0; p < n / T; p++) {   // step through the tiles along the row and column
        As[threadIdx.y][threadIdx.x] = A[row * n + p * T + threadIdx.x];   // each thread loads ONE number of A's tile
        Bs[threadIdx.y][threadIdx.x] = B[(p * T + threadIdx.y) * n + col];   // and ONE number of B's tile
        __syncthreads();   // wait until the whole block has loaded both tiles
        for (int k = 0; k < T; k++)   // then T multiply-adds...
            sum += As[threadIdx.y][k] * Bs[k][threadIdx.x];   // ...all from shared memory
        __syncthreads();   // wait before the tiles are overwritten
    }
    C[row * n + col] = sum;   // write the finished cell
}
'''
mod = cp.RawModule(code=mm_src)   # compile both kernels
naive, tiled = mod.get_function("matmul_naive"), mod.get_function("matmul_tiled")   # get each one by name
tile_res = {}   # (n, contestant) → TFLOPS, for the chart
for n in [1024, 2048]:   # two sizes, both multiples of 16
    A = cp.random.rand(n, n, dtype=cp.float32); B = cp.random.rand(n, n, dtype=cp.float32)   # two random matrices on the GPU
    C = cp.empty((n, n), dtype=cp.float32)   # space for the answer
    grid, block = (n // 16, n // 16), (16, 16)   # one thread per output cell, 16 × 16 threads per block
    flops = 2 * n**3   # the work in one n × n multiply
    t_naive = cuda_time_ms(lambda: naive(grid, block, (A, B, C, cp.int32(n))), repeats=3)   # time the naive kernel
    ok_naive = bool(cp.allclose(C, A @ B, rtol=1e-3, atol=1e-2))   # check its answer (allowing for rounding)
    t_tiled = cuda_time_ms(lambda: tiled(grid, block, (A, B, C, cp.int32(n))), repeats=3)   # time the tiled kernel
    ok_tiled = bool(cp.allclose(C, A @ B, rtol=1e-3, atol=1e-2))   # check its answer
    At, Bt = torch.as_tensor(A, device="cuda"), torch.as_tensor(B, device="cuda")   # the same data as PyTorch tensors
    t_cublas = timeit(lambda: At @ Bt) * 1e3   # PyTorch's @ calls NVIDIA's cuBLAS library (× 1e3 → ms)
    Ah, Bh = At.half(), Bt.half()   # .half() converts to FP16
    t_fp16 = timeit(lambda: Ah @ Bh) * 1e3   # an FP16 multiply runs on the Tensor Cores
    print(f"n = {n}")   # a heading for this size
    for name, t, ok in [("naive kernel", t_naive, ok_naive), ("tiled kernel", t_tiled, ok_tiled), ("cuBLAS FP32", t_cublas, True), ("cuBLAS FP16 (Tensor Cores)", t_fp16, True)]:   # one line per contestant
        tile_res[(n, name)] = flops / (t / 1e3) / 1e12   # its speed in TFLOPS
        print(f"   {name:27s} {t:8.2f} ms  {tile_res[(n, name)]:6.2f} TFLOPS   correct={ok}")   # time, speed, and the check

### ✏️ YOUR TURN (A)
1. How many times faster was the tiled kernel than the naive one? Which idea from the lesson explains it?
2. Change `#define T 16` to `8` (and the block/grid to 8 × 8 and `n // 8`). Faster or slower? Why?

*Your answers:*

---
## Part B · Precision  *(Unit 5.2)*

**Speed and error.** Multiply in FP32 and FP16, and compare both with an FP64 (very precise) reference.

In [ ]:
n = 2048   # 2048 × 2048 matrices
A = torch.rand(n, n, device=device, dtype=torch.float64); B = torch.rand(n, n, device=device, dtype=torch.float64)   # FP64: 64-bit, very precise
ref = A @ B   # our "true" answer to compare against
for dtype in [torch.float32, torch.float16]:   # 32-bit, then 16-bit
    a, b = A.to(dtype), B.to(dtype)   # the same matrices, rounded to this format
    t = timeit(lambda: a @ b, repeats=10)   # time one multiply
    err = ((a @ b).double() - ref).abs().max() / ref.abs().max()   # the biggest error, relative to the size of the answer
    print(f"{str(dtype):14s}: {t*1e3:7.2f} ms  {2*n**3/t/1e12:6.2f} TFLOPS   max relative error {err:.1e}")   # time, speed and error

**Range: overflow and underflow.** FP16's largest number is 65,504; BF16 keeps FP32's range.

In [ ]:
for v in [3.14159265, 70000.0, 2e-8]:   # an ordinary number, a big one and a tiny one
    row = [f"{v:>12g}"]   # start the printed row with the original value
    for dt in [torch.float32, torch.float16, torch.bfloat16]:   # three number formats
        row.append(f"{str(dt)[6:]}: {torch.tensor(v, dtype=dt).item():.6g}")   # store v in this format, read it back
    print("   ".join(row))   # show what survived in each format

70000 becomes `inf` in FP16 (overflow) and 2e-8 becomes 0 (underflow); BF16 keeps both, with fewer digits.

**Mixed-precision training.** The same wide network on MNIST (the handwritten digits from Lab 1), one epoch with FP32 and one with `autocast` + `GradScaler`. To measure the GPU work itself, the data is kept on the GPU.

In [ ]:
from torchvision import datasets   # ready-made datasets, including MNIST
mnist = datasets.MNIST("data", train=True, download=True)   # the 60,000 handwritten training digits
mnist_test = datasets.MNIST("data", train=False, download=True)   # and 10,000 test digits
Xtr = (mnist.data.float() / 255).view(-1, 784).to(device); Ytr = mnist.targets.to(device)   # all images as rows of 784 pixels (0..1), and labels, on the GPU
Xte = (mnist_test.data.float() / 255).view(-1, 784).to(device); Yte = mnist_test.targets.to(device)   # the same for the test set

def wide_net():   # a wide network (2,048 neurons per layer), so the GPU has plenty of work
    torch.manual_seed(0)   # the same starting weights every time: a fair comparison
    return nn.Sequential(nn.Linear(784, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10)).to(device)   # 784 → 2048 → 2048 → 10, on the GPU

def accuracy(model, X, Y):   # the fraction of images whose highest score is the right digit
    with torch.no_grad():   # no training here, so keep no history
        return (model(X).argmax(1) == Y).float().mean().item()   # argmax(1) = the class with the highest score

def train_epoch(model, X, Y, batch=256, amp=False, lr=0.1):   # one epoch; amp=True turns on mixed precision
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)   # gradient descent with momentum
    scaler = torch.amp.GradScaler("cuda", enabled=amp)   # loss scaling (does nothing when amp is off)
    lossf = nn.CrossEntropyLoss()   # the loss for "which class?" problems
    for idx in torch.randperm(len(X), device=device).split(batch):   # shuffle the image numbers, cut them into batches
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):   # inside this block, big multiplies run in FP16
            loss = lossf(model(X[idx]), Y[idx])   # forward pass and loss for this batch
        opt.zero_grad(set_to_none=True)   # clear the old slopes
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()   # scale the loss up, backward, update, adjust the scale

mixed = {}   # amp on/off → (seconds, accuracy)
for amp in [False, True]:   # first plain FP32, then mixed precision
    model = wide_net()   # a fresh network
    sync(); t0 = time.perf_counter(); train_epoch(model, Xtr, Ytr, amp=amp); sync()   # time one epoch
    mixed[amp] = (time.perf_counter() - t0, accuracy(model, Xte, Yte))   # keep the time and the test accuracy
    print(f"{'mixed precision' if amp else 'FP32           '}: {mixed[amp][0]:6.2f} s per epoch, test accuracy {mixed[amp][1]:.2%}")   # show both

### ✏️ YOUR TURN (B)
1. How much faster was FP16 for the matrix multiply? How big was its error?
2. Did mixed precision change the accuracy? Explain using Unit 5.2's "Does AI notice?".

*Your answers:*

---
## Part C · Your Own Roofline  *(Unit 5.3)*

First measure **this GPU's roof** (its best FP32 speed and its best bandwidth). Then measure each operation's FLOPs, bytes and time, and place it under the roof. (On a T4 expect a roof of roughly 4–5 TFLOPS measured against 8.1 TFLOPS published, and roughly 250–300 GB/s against 320 GB/s published.)

In [ ]:
W4 = torch.rand(4096, 4096, device=device)   # a big matrix, used again below
PEAK = max(2 * m**3 / timeit(lambda: torch.rand(m, m, device=device) @ torch.rand(m, m, device=device), repeats=5) for m in [1024, 2048, 4096])   # this GPU's best FP32 speed over three sizes
big = torch.rand(1 << 27, device=device)   # 512 MB, for measuring its bandwidth
BW = 2 * big.numel() * 4 / timeit(lambda: big.clone(), repeats=5)   # its best bandwidth: a big copy (read + write)
print(f"This GPU's roof: {PEAK / 1e12:.1f} TFLOPS and {BW / 1e9:.0f} GB/s, so the ridge is at {PEAK / BW:.0f} FLOPs per byte\n")   # show the roof
results = []   # (name, intensity, achieved FLOPS) for every operation we measure

def record(name, fn, flops, nbytes):   # time one operation and store its intensity and speed
    t = timeit(fn, repeats=10)   # seconds per run
    results.append((name, flops / nbytes, flops / t))   # intensity = FLOPs per byte; speed = FLOPs per second
    print(f"{name:16s} intensity {flops/nbytes:8.2f} FLOPs/B   achieved {flops/t/1e9:9.1f} GFLOPS   ({nbytes/t/1e9:6.1f} GB/s)")   # show both, and the bandwidth

n = 50_000_000   # 5 crore numbers
a = torch.rand(n, device=device); b = torch.rand(n, device=device)   # two big arrays on the GPU
record("vector add", lambda: a + b, n, 12 * n)   # n adds; read a and b, write the answer: 3 × 4 bytes each
record("ReLU", lambda: torch.relu(a), n, 8 * n)   # n comparisons; read and write: 2 × 4 bytes each
x1 = torch.rand(1, 4096, device=device)   # one input of 4,096 numbers
record("layer, batch 1", lambda: x1 @ W4, 2 * 4096 * 4096, 4 * (4096 * 4096 + 2 * 4096))   # 2·4096² FLOPs, but must read all the weights
for m in [256, 1024, 4096]:   # three square matrix multiplies
    A = torch.rand(m, m, device=device); B = torch.rand(m, m, device=device)   # two m × m matrices
    record(f"matmul {m}", lambda: A @ B, 2 * m**3, 12 * m * m)   # 2m³ FLOPs; 3 matrices × m² × 4 bytes

In [ ]:
I = np.logspace(-2, 4, 200)                        # 200 intensities from 0.01 to 10,000, evenly spaced on a log scale
plt.figure(figsize=(7, 4.5))                       # a new chart
plt.loglog(I, np.minimum(PEAK, BW * I), "k-", lw=2, label="this GPU's roofline (FP32)")   # the roof: min(peak, bandwidth × intensity)
for name, inten, perf in results:                  # every operation we measured
    plt.loglog(inten, perf, "o", ms=8)             # one dot per operation (ms = marker size)
    plt.annotate(name, (inten, perf), textcoords="offset points", xytext=(6, -10), fontsize=8)   # its name beside it
plt.axvline(PEAK / BW, ls=":", c="gray")           # a dotted vertical line at the ridge point
plt.text(PEAK / BW * 1.1, 1e9, f"ridge ≈ {PEAK/BW:.0f}", fontsize=8)   # label it
plt.xlabel("arithmetic intensity (FLOPs per byte)"); plt.ylabel("achieved FLOPS")   # axis labels
plt.grid(alpha=0.3, which="both"); plt.legend()    # faint grid and the colour key
plt.title("Your measurements under your GPU's roofline"); plt.show()   # title, then draw it

### ✏️ YOUR TURN (C)
1. Which operations sat close to the slanted (memory) roof, and which near the flat (compute) roof?
2. Use your "layer, batch 1" bandwidth to estimate tokens per second for a 1-billion-parameter FP16 chatbot on your GPU.

*Your answers:*

---
## Part D · Batch Size and Training on the GPU  *(Unit 5.4)*

**CPU vs GPU** for the wide network: time 100 training steps on each.

In [ ]:
def steps_time(dev, steps=100, batch=256):   # time training steps of the wide network on device dev
    torch.manual_seed(0)   # the same starting weights every time
    net = nn.Sequential(nn.Linear(784, 2048), nn.ReLU(), nn.Linear(2048, 2048), nn.ReLU(), nn.Linear(2048, 10)).to(dev)   # the wide network on dev
    opt = torch.optim.SGD(net.parameters(), lr=0.05); lossf = nn.CrossEntropyLoss()   # optimiser and loss
    X = torch.rand(batch, 784, device=dev); Y = torch.randint(0, 10, (batch,), device=dev)   # one fixed batch of made-up data
    def run():   # the steps themselves
        for _ in range(steps):   # one training step each time round
            loss = lossf(net(X), Y); opt.zero_grad(); loss.backward(); opt.step()   # forward, clear slopes, backward, update
    run()   # a warm-up run
    if dev == "cuda": torch.cuda.synchronize()   # wait for the GPU before starting the clock
    t0 = time.perf_counter(); run()   # the timed run
    if dev == "cuda": torch.cuda.synchronize()   # and wait again before stopping it
    return time.perf_counter() - t0   # seconds for all the steps

t_cpu = steps_time("cpu", steps=20) * 5   # 20 steps on the CPU, scaled up to 100 (the CPU is slow!)
t_gpu = steps_time(device)   # 100 steps on the GPU
print(f"CPU: {t_cpu:6.2f} s per 100 steps")   # show both...
print(f"GPU: {t_gpu:6.2f} s per 100 steps   → {t_cpu / t_gpu:.1f}× faster")   # ...and the speed-up

**Batch-size sweep:** throughput (images per second) and peak GPU memory.

In [ ]:
batches, tput, mem = [16, 64, 256, 1024, 4096], [], []   # batch sizes to try; results go in tput and mem
for B in batches:   # for each batch size
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()   # free cached memory and reset the "highest so far" counter
    # time 30 training steps, three times, and keep the best (a GPU's clock speed jumps about)
    t = min(steps_time(device, steps=30, batch=B) for _ in range(3))
    tput.append(30 * B / t)   # throughput = images processed ÷ seconds
    mem.append(torch.cuda.max_memory_allocated() / 1e9)   # the most GPU memory used, in GB
    print(f"batch {B:5d}: {tput[-1]:10,.0f} images/s   peak memory {mem[-1]:.2f} GB")   # show both

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))                 # two charts side by side: ax[0] and ax[1]
ax[0].semilogx(batches, tput, "o-"); ax[0].set_xlabel("batch size"); ax[0].set_ylabel("images / s")     # throughput
ax[0].set_title("Throughput rises, then levels off"); ax[0].grid(alpha=0.3)                            # its title and grid
ax[1].semilogx(batches, mem, "s-", c="tab:red"); ax[1].set_xlabel("batch size"); ax[1].set_ylabel("GB") # memory
ax[1].set_title("Memory keeps climbing"); ax[1].grid(alpha=0.3)                                        # its title and grid
plt.tight_layout(); plt.show()                                  # stop them overlapping, then draw

### ✏️ YOUR TURN (D)
1. How much faster was the GPU for the wide network? Compare with the small Lab 1 network (Unit 5.4 explains the difference).
2. Where did throughput level off? Which batch size would you choose, and why?

*Your answers:*

---
## Part E · The Speed-Up Ladder  *(Unit 5.5)*

**What is Fashion-MNIST?** A harder twin of MNIST, released by the online shop Zalando in 2017. Same format — 70,000 grey 28 × 28 pictures (60,000 training + 10,000 test), 10 classes — but the pictures are **clothing** instead of digits: T-shirt/top, trouser, pullover, dress, coat, sandal, shirt, sneaker, bag, ankle boot. Telling a shirt from a pullover is harder than telling a 1 from a 7, so accuracy is lower (about 80–88% for a simple network instead of 97%).

Network 784 → 1024 → 1024 → 10. Each rung adds ONE change. We time one epoch and check test accuracy at every rung.

In [ ]:
from torchvision import transforms   # ways to convert images, e.g. into tensors
from torch.utils.data import DataLoader   # serves a dataset in batches
fash = datasets.FashionMNIST("data", train=True, download=True, transform=transforms.ToTensor())   # 60,000 clothing pictures, as tensors
fash_test = datasets.FashionMNIST("data", train=False, download=True)   # 10,000 test pictures
FXte = (fash_test.data.float() / 255).view(-1, 784).to(device); FYte = fash_test.targets.to(device)   # the test set as one tensor on the GPU
FX = (fash.data.float() / 255).view(-1, 784).to(device); FY = fash.targets.to(device)   # the training set too (used by rung 5)

# the six rungs of the ladder, one change each
RUNGS = ["0 baseline: batch 32, .item() every step", "1 + no per-step sync", "2 + num_workers=2, pin_memory, non_blocking",
         "3 + batch 256 (lr scaled up)", "4 + mixed precision", "5 + whole dataset on the GPU"]

def ladder(rung):   # train one epoch with every improvement up to this rung switched on
    torch.manual_seed(0)   # the same starting weights on every rung
    model = nn.Sequential(nn.Flatten(), nn.Linear(784, 1024), nn.ReLU(), nn.Linear(1024, 1024), nn.ReLU(), nn.Linear(1024, 10)).to(device)   # 784 → 1024 → 1024 → 10
    B = 256 if rung >= 3 else 32   # rung 3 and up: a bigger batch
    opt = torch.optim.SGD(model.parameters(), lr=0.2 if rung >= 3 else 0.05, momentum=0.9)   # with a bigger learning rate
    amp = rung >= 4   # rung 4 and up: mixed precision
    scaler = torch.amp.GradScaler("cuda", enabled=amp); lossf = nn.CrossEntropyLoss()   # loss scaling and the loss
    if rung >= 5:   # rung 5: the data is already on the GPU
        batches = ((FX[i], FY[i]) for i in torch.randperm(len(FX), device=device).split(B))   # just shuffle and slice it: no DataLoader, no copies
    else:   # rungs 0–4 use a DataLoader
        workers = 2 if rung >= 2 else 0   # rung 2 and up: two helper processes...
        batches = DataLoader(fash, batch_size=B, shuffle=True, num_workers=workers, pin_memory=(rung >= 2))   # ...and pinned memory
    log, running = [], torch.zeros((), device=device)   # a Python list (rung 0) or a total kept on the GPU (rung 1 and up)
    sync(); t0 = time.perf_counter()   # start timing the epoch
    for x, y in batches:   # one batch at a time
        x = x.to(device, non_blocking=True); y = y.to(device, non_blocking=True)   # copy it to the GPU (does nothing if already there)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp):   # FP16 inside this block, if amp is on
            loss = lossf(model(x), y)   # forward pass and loss
        opt.zero_grad(set_to_none=True)   # clear the old slopes
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()   # backward and update
        if rung == 0:   # the baseline...
            log.append(loss.item())   # ...waits for the GPU every step
        else:   # rung 1 and up...
            running += loss.detach()   # ...keep the total on the GPU: no waiting
    sync(); seconds = time.perf_counter() - t0   # stop the clock
    return seconds, accuracy(model, FXte, FYte)   # the epoch's time, and the test accuracy

rung_res = []   # (seconds, accuracy) for each rung
for r, name in enumerate(RUNGS):   # climb the ladder, rung 0 to 5
    rung_res.append(ladder(r))   # train one epoch on this rung
    s, acc = rung_res[-1]   # its time and accuracy
    print(f"{name:46s} {s:6.2f} s   {rung_res[0][0] / s:5.1f}× faster   test accuracy {acc:.1%}")   # compare with rung 0

In [ ]:
names = ["T-shirt", "trouser", "pullover", "dress", "coat", "sandal", "shirt", "sneaker", "bag", "boot"]   # the 10 classes, in order
plt.figure(figsize=(10, 1.6))                                    # one wide, short drawing area
for i in range(10):                                              # the first 10 test pictures
    plt.subplot(1, 10, i + 1)                                    # 1 row of 10 boxes: draw in box i+1
    plt.imshow(fash_test.data[i], cmap="gray")                   # test picture i
    plt.title(names[fash_test.targets[i]], fontsize=8)           # its label, as a word
    plt.axis("off")                                              # no axes around the picture
plt.show()                                                       # draw them

Not every rung helps on every machine, and that is the real lesson. After one epoch, a wobble of 2–4% in accuracy between rungs is normal noise from different batch orders and sizes: re-run a rung to check before believing it.

### ✏️ YOUR TURN (E) — record your results
| Rung | Seconds per epoch | Speed-up | Test accuracy |
|---|---|---|---|
| 0 baseline | | 1.0× | |
| 1 no sync | | | |
| 2 workers + pinned | | | |
| 3 batch 256 | | | |
| 4 mixed precision | | | |
| 5 data on GPU | | | |

1. Which single rung gave the biggest jump? Explain why with a unit you studied.
2. Did any rung change the accuracy noticeably?
3. Did you reach the 5× target?

*Your answers:*

---
## 🎓 Wrap-up
- Tiling reuses data from shared memory; libraries like cuBLAS go further.
- FP16/BF16/INT8 halve or quarter the bytes and unlock Tensor Cores, and AI barely notices.
- The roofline tells you whether memory or compute is the limit, and which fix will work.
- Batch size, feeding the GPU and mixed precision together turn a slow run into a fast one.
- **Next — Lab 5B (second half of Unit 5): GPUs in Real AI Projects.** The software stack, sizing memory, out-of-memory errors, and many GPUs.